### **Clasificación multimodal con transformers: implementación de MMBT-Lite con BERT preentrenado**

Este cuaderno desarrolla una implementación de **MMBT-Lite** para clasificación multimodal supervisada, inspirada en la idea central de [Kiela et al. (2019)](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Kiela.pdf): representar **imagen y texto dentro de una secuencia conjunta** y modelar su interacción mediante **self-attention** en una arquitectura tipo transformer. 

Desde el punto de vista técnico, el cuaderno aborda cuatro componentes centrales del modelado multimodal contemporáneo:

* **encoder textual preentrenado** basado en `bert-base-uncased`,
* **encoder visual ligero** basado en `EfficientNet-B0`,
* **proyección de tokens visuales** al espacio oculto del transformer textual,
* **fine-tuning conjunto** con **tasas de aprendizaje discriminativos** para texto, visión y capas nuevas.

La tarea experimental se formula como un problema supervisado de **clasificación binaria match /mismatch (correspondencia/no correspondencia)** entre imagen y caption. Para evitar una tarea artificialmente trivial, los ejemplos negativos se construyen mediante **TF-IDF + nearest neighbors**, generando pares incompatibles pero semánticamente cercanos. Esto permite evaluar de manera más informativa la capacidad del modelo para integrar evidencia visual y textual.

El cuaderno compara dos familias de arquitectura:

1. **Fusión tardía**, donde texto e imagen se codifican por separado y se combinan únicamente al final.
2. **MMBT-Lite**, donde ambas modalidades se integran **dentro del encoder**, permitiendo interacción multimodal temprana y distribuida a través de self-attention.

En términos computacionales, el diseño mantiene un equilibrio entre **rigurosidad conceptual** y **costo razonable**, de modo que pueda ejecutarse en entornos como Colab con CPU o GPU. El interés principal no es reproducir exactamente la configuración experimental del paper, sino estudiar con fidelidad su intuición arquitectónica: el paso desde fusiones externas y manuales hacia una **deep fusion multimodal (fusión multimodal profunda) basada en transformers**. 




In [ ]:
# Si estás en Colab o en un entorno limpio, descomenta esta celda.
# %pip install -q -U transformers datasets accelerate evaluate scikit-learn pillow matplotlib tqdm


In [ ]:
import math
import random
import copy
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import load_dataset
from transformers import BertTokenizerFast, BertModel, get_linear_schedule_with_warmup
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def resolve_device(preference: str = "auto") -> torch.device:
    preference = preference.lower().strip()
    if preference == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if preference == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA no está disponible en este entorno. Se usará CPU.")
        return torch.device("cpu")
    if preference == "cpu":
        return torch.device("cpu")
    raise ValueError("device_preference debe ser 'auto', 'cpu' o 'cuda'.")

In [ ]:
CONFIG = {
    "dataset_name": "jxie/flickr8k",
    "max_images_train": 600,      # 600 imágenes -> 3000 captions positivas
    "max_images_val": 150,
    "max_images_test": 150,
    "max_text_len": 40,
    "batch_size": 16,
    "epochs": 3,
    "device_preference": "auto",  # opciones: "auto", "cpu", "cuda"
    "num_workers": 0,             # en notebook: 0 para evitar errores de workers
    "use_amp": True,              # mixed precision solo si DEVICE == cuda
    "visual_grid": (3, 3),        # 9 tokens visuales
    "dropout": 0.1,
    "lr_text": 2e-5,
    "lr_vision": 8e-5,
    "lr_head": 2e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "patience": 2,
    "run_late_fusion": True,
    "run_mmbt": True,
}

DEVICE = resolve_device(CONFIG["device_preference"])
USE_CUDA = DEVICE.type == "cuda"
USE_AMP = bool(CONFIG["use_amp"] and USE_CUDA)

PIN_MEMORY = USE_CUDA

runtime_info = {
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "amp_enabled": USE_AMP,
    "num_workers": CONFIG["num_workers"],
    "pin_memory": PIN_MEMORY,
}
if USE_CUDA:
    runtime_info["gpu_name"] = torch.cuda.get_device_name(0)

print(runtime_info)
CONFIG

##### **Selección de CPU o GPU**

Este cuaderno ahora permite tres modos en `CONFIG["device_preference"]`:

- `"auto"`: usa GPU si hay CUDA disponible; si no, usa CPU.
- `"cuda"`: intenta usar GPU; si no existe, hace *fallback* a CPU.
- `"cpu"`: fuerza ejecución en CPU.

Además:

- `CONFIG["use_amp"] = True` activa *mixed precision* solo cuando el dispositivo elegido es CUDA.
- `CONFIG["num_workers"] = None` deja que el cuaderno ajuste automáticamente los `DataLoader` según el dispositivo.

#### **1. Carga del dataset real**

Usamos **Flickr8k**, que contiene imágenes y cinco captions por imagen. En vez de hacer captioning o retrieval (recuperación), convertimos el problema en una **clasificación binaria multimodal**:
- `1`: caption correcto para la imagen
- `0`: caption incorrecto pero razonablemente parecido a otros captions del corpus

Eso nos permite mantener la estructura de **clasificación multimodal** del paper, sin replicar exactamente sus datasets.


In [ ]:
raw = load_dataset(CONFIG["dataset_name"])
raw


In [ ]:
def build_caption_table(hf_split, max_images: Optional[int], seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    all_indices = np.arange(len(hf_split))
    if max_images is not None and max_images < len(all_indices):
        all_indices = rng.choice(all_indices, size=max_images, replace=False)
        all_indices = np.sort(all_indices)

    rows = []
    for idx in tqdm(all_indices, desc="Exploding captions"):
        ex = hf_split[int(idx)]
        for j in range(5):
            cap_key = f"caption_{j}"
            if cap_key in ex and ex[cap_key] is not None:
                rows.append({
                    "hf_index": int(idx),
                    "image_id": int(idx),
                    "caption": ex[cap_key].strip(),
                })
    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return df

train_meta = build_caption_table(raw["train"], CONFIG["max_images_train"], seed=SEED)
val_meta   = build_caption_table(raw["validation"], CONFIG["max_images_val"], seed=SEED + 1)
test_meta  = build_caption_table(raw["test"], CONFIG["max_images_test"], seed=SEED + 2)

train_meta.head(), len(train_meta), len(val_meta), len(test_meta)


#### **2. Construcción de negativos semi-difíciles**

En lugar de negativos completamente aleatorios, buscamos **descripciones textuales (captions)** lexicalmente parecidas con TF-IDF y vecinos más cercanos.
La idea es que el modelo no gane solo por detectar desajustes obvios, queremos una tarea algo más fina.


In [ ]:
def build_binary_match_dataset(meta_df: pd.DataFrame, seed: int = 42, topk: int = 15) -> pd.DataFrame:
    rng = random.Random(seed)
    df = meta_df.copy().reset_index(drop=True)

    # Positivos
    pos = df.copy()
    pos["label"] = 1
    pos["source"] = "positive"

    # TF-IDF para buscar captions cercanos pero de otra imagen
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=12000,
        ngram_range=(1, 2),
        min_df=1,
    )
    X = vectorizer.fit_transform(df["caption"].tolist())
    nn_model = NearestNeighbors(
        n_neighbors=min(topk + 20, len(df)),
        metric="cosine",
        algorithm="brute"
    )
    nn_model.fit(X)
    _, neighbors = nn_model.kneighbors(X)

    neg_rows = []
    image_ids = df["image_id"].tolist()

    for i, nbrs in enumerate(tqdm(neighbors, desc="Mining negatives")):
        # filtramos captions de otra imagen
        candidates = [j for j in nbrs[1:] if image_ids[j] != image_ids[i]]

        if not candidates:
            # fallback
            candidates = [j for j in range(len(df)) if image_ids[j] != image_ids[i]]
            j = rng.choice(candidates)
        else:
            j = rng.choice(candidates[:min(topk, len(candidates))])

        neg_rows.append({
            "hf_index": int(df.loc[i, "hf_index"]),
            "image_id": int(df.loc[i, "image_id"]),
            "caption": str(df.loc[j, "caption"]),
            "label": 0,
            "source": "hard_negative",
            "neg_caption_from_image_id": int(df.loc[j, "image_id"]),
        })

    neg = pd.DataFrame(neg_rows)

    out = pd.concat([pos, neg], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

train_pairs = build_binary_match_dataset(train_meta, seed=SEED, topk=15)
val_pairs   = build_binary_match_dataset(val_meta, seed=SEED + 1, topk=10)
test_pairs  = build_binary_match_dataset(test_meta, seed=SEED + 2, topk=10)

train_pairs.head(), train_pairs["label"].value_counts().to_dict()


In [ ]:
print("train:", train_pairs.shape, " | positives:", int((train_pairs.label == 1).sum()), " negatives:", int((train_pairs.label == 0).sum()))
print("val:  ", val_pairs.shape)
print("test: ", test_pairs.shape)

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
train_pairs["source"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Fuentes de ejemplos (train)")
ax.set_ylabel("count")
plt.show()


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
vision_weights = EfficientNet_B0_Weights.DEFAULT
vision_transform = vision_weights.transforms()

class FlickrMatchDataset(Dataset):
    def __init__(self, hf_split, pair_df: pd.DataFrame, image_transform):
        self.hf_split = hf_split
        self.df = pair_df.reset_index(drop=True)
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        example = self.hf_split[int(row.hf_index)]
        image = example["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        pixel_values = self.image_transform(image.convert("RGB"))
        return {
            "pixel_values": pixel_values,
            "caption": str(row.caption),
            "label": int(row.label),
            "hf_index": int(row.hf_index),
            "image_id": int(row.image_id),
            "source": row.source,
        }

def collate_fn(batch):
    captions = [x["caption"] for x in batch]
    enc = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=CONFIG["max_text_len"],
        return_tensors="pt",
    )
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
    meta = {
        "hf_index": [x["hf_index"] for x in batch],
        "image_id": [x["image_id"] for x in batch],
        "caption": captions,
        "source": [x["source"] for x in batch],
    }
    return {
        "pixel_values": pixel_values,
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": labels,
        "meta": meta,
    }

train_ds = FlickrMatchDataset(raw["train"], train_pairs, vision_transform)
val_ds   = FlickrMatchDataset(raw["validation"], val_pairs, vision_transform)
test_ds  = FlickrMatchDataset(raw["test"], test_pairs, vision_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

batch = next(iter(train_loader))
{k: (v.shape if torch.is_tensor(v) else type(v)) for k, v in batch.items() if k != "meta"}

In [ ]:
# Inspección rápida de un ejemplo positivo y uno negativo
def show_examples(ds: FlickrMatchDataset, n: int = 4):
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.array([axes])

    shown = 0
    idx = 0
    while shown < n and idx < len(ds):
        item = ds[idx]
        if shown < n:
            img = raw["train"][item["hf_index"]]["image"] if ds.hf_split is raw["train"] else ds.hf_split[item["hf_index"]]["image"]
            axes[shown, 0].imshow(img)
            axes[shown, 0].axis("off")
            axes[shown, 0].set_title(f"label={item['label']} | {item['source']}")

            axes[shown, 1].axis("off")
            axes[shown, 1].text(0, 0.7, item["caption"], fontsize=12, wrap=True)
            axes[shown, 1].set_title("caption")
            shown += 1
        idx += 1

    plt.tight_layout()
    plt.show()

show_examples(train_ds, n=4)


#### **3. Modelos**

En esta sección se comparan dos estrategias de clasificación multimodal que difieren en el punto donde ocurre la interacción entre imagen y texto.

**A. Fusión tardía**

Se utiliza como baseline fuerte y conceptualmente claro. La arquitectura combina:

* **BERT preentrenado** como encoder textual,
* **EfficientNet-B0 preentrenado** como encoder visual,
* **concatenación final** de ambas representaciones,
* **clasificador MLP** sobre la representación fusionada.

En este esquema, cada modalidad se procesa de manera independiente y la fusión ocurre únicamente en la etapa de decisión. Por ello, funciona como referencia útil para contrastar arquitecturas con integración multimodal más profunda.

**B. MMBT-Lite**

Esta variante sigue de manera más directa la lógica de MMBT. En lugar de resumir la imagen en un único vector global, el modelo construye una representación visual tokenizada y la integra con el texto dentro del transformer. El procedimiento es el siguiente:

* se extrae un **mapa espacial de activaciones** del encoder visual,
* ese mapa se reduce a una cuadrícula `3×3`, obteniendo **9 tokens visuales**,
* cada token visual se **proyecta al hidden size de BERT**,
* los tokens visuales se concatenan con la secuencia textual,
* la secuencia multimodal completa se procesa con el **encoder de BERT**,
* la clasificación final se realiza a partir del estado **`[CLS]`**.

La diferencia clave frente a Late Fusion es que aquí la interacción entre modalidades no queda restringida a la última capa, sino que ocurre dentro del encoder mediante **self-attention** entre tokens textuales y visuales.

En esta implementación no se emplea una estrategia de **freeze/unfreeze**. En su lugar, se realiza **fine-tuning conjunto total**, usando **tasas de aprendizaje discriminativos** para ajustar de forma diferenciada el encoder textual, el encoder visual y las capas nuevas de proyección y clasificación.



In [ ]:
class LateFusionClassifier(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", num_labels=2, dropout=0.1):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained(bert_name)
        effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.image_features = effnet.features
        self.image_pool = nn.AdaptiveAvgPool2d((1, 1))
        vision_dim = 1280
        hidden = self.text_encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden + vision_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels),
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        text_cls = text_out.last_hidden_state[:, 0]

        feat_map = self.image_features(pixel_values)
        img_vec = self.image_pool(feat_map).flatten(1)

        fused = torch.cat([text_cls, img_vec], dim=-1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}


class MMBTLiteClassifier(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", visual_grid=(3, 3), num_labels=2, dropout=0.1):
        super().__init__()
        self.text_encoder = BertModel.from_pretrained(bert_name)
        effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.image_features = effnet.features

        self.visual_grid = visual_grid
        self.visual_pool = nn.AdaptiveAvgPool2d(visual_grid)
        self.visual_dim = 1280
        self.hidden = self.text_encoder.config.hidden_size
        self.num_visual_tokens = visual_grid[0] * visual_grid[1]

        self.visual_proj = nn.Linear(self.visual_dim, self.hidden)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden, self.hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.hidden, num_labels),
        )

    def _build_text_embeddings(self, input_ids):
        B, L = input_ids.shape
        device = input_ids.device
        pos_ids = torch.arange(L, device=device).unsqueeze(0).expand(B, -1)
        tok_type_ids = torch.zeros((B, L), dtype=torch.long, device=device)

        word = self.text_encoder.embeddings.word_embeddings(input_ids)
        pos = self.text_encoder.embeddings.position_embeddings(pos_ids)
        typ = self.text_encoder.embeddings.token_type_embeddings(tok_type_ids)
        return word + pos + typ

    def _build_visual_embeddings(self, visual_tokens):
        B, N, _ = visual_tokens.shape
        device = visual_tokens.device
        pos_ids = torch.arange(N, device=device).unsqueeze(0).expand(B, -1)
        tok_type_ids = torch.ones((B, N), dtype=torch.long, device=device)

        vis = self.visual_proj(visual_tokens)
        pos = self.text_encoder.embeddings.position_embeddings(pos_ids)
        typ = self.text_encoder.embeddings.token_type_embeddings(tok_type_ids)
        return vis + pos + typ

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, output_attentions=False):
        B = input_ids.size(0)

        # Texto -> embeddings BERT
        text_embeds = self._build_text_embeddings(input_ids)

        # Imagen -> mapa espacial -> tokens visuales
        feat_map = self.image_features(pixel_values)                   # [B, 1280, H, W]
        pooled = self.visual_pool(feat_map)                            # [B, 1280, gh, gw]
        visual_tokens = pooled.flatten(2).transpose(1, 2).contiguous() # [B, N, 1280]
        visual_embeds = self._build_visual_embeddings(visual_tokens)   # [B, N, hidden]
        N = visual_embeds.size(1)

        # Secuencia multimodal
        embeds = torch.cat([text_embeds, visual_embeds], dim=1)        # [B, L+N, hidden]
        embeds = self.text_encoder.embeddings.LayerNorm(embeds)
        embeds = self.text_encoder.embeddings.dropout(embeds)

        multimodal_mask = torch.cat(
            [
                attention_mask,
                torch.ones((B, N), dtype=attention_mask.dtype, device=attention_mask.device),
            ],
            dim=1,
        )

        ext_mask = self.text_encoder.get_extended_attention_mask(
            multimodal_mask,
            multimodal_mask.shape,
        )

        encoder_out = self.text_encoder.encoder(
            embeds,
            attention_mask=ext_mask,
            output_attentions=output_attentions,
            return_dict=True,
        )

        last_hidden = encoder_out.last_hidden_state
        cls = self.dropout(last_hidden[:, 0])
        logits = self.classifier(cls)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        out = {
            "loss": loss,
            "logits": logits,
            "last_hidden_state": last_hidden,
            "visual_tokens": visual_tokens,
            "multimodal_mask": multimodal_mask,
        }
        if output_attentions:
            out["attentions"] = encoder_out.attentions
        return out

In [ ]:
@torch.no_grad()
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

late_model = LateFusionClassifier(dropout=CONFIG["dropout"]).to(DEVICE)
mmbt_model = MMBTLiteClassifier(
    visual_grid=CONFIG["visual_grid"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

print("LateFusion trainable params:", f"{count_trainable_params(late_model):,}")
print("MMBT-Lite trainable params:", f"{count_trainable_params(mmbt_model):,}")


In [ ]:
def make_optimizer_and_scheduler(model, train_loader_len, epochs):
    no_decay = ["bias", "LayerNorm.weight"]

    def split_decay(named_params, lr):
        decay_params, no_decay_params = [], []
        for n, p in named_params:
            if not p.requires_grad:
                continue
            if any(nd in n for nd in no_decay):
                no_decay_params.append(p)
            else:
                decay_params.append(p)
        return [
            {"params": decay_params, "lr": lr, "weight_decay": CONFIG["weight_decay"]},
            {"params": no_decay_params, "lr": lr, "weight_decay": 0.0},
        ]

    optimizer_groups = []
    optimizer_groups += split_decay(model.text_encoder.named_parameters(), CONFIG["lr_text"])
    optimizer_groups += split_decay(model.image_features.named_parameters(), CONFIG["lr_vision"])

    head_named = []
    for name, module in model.named_children():
        if name not in {"text_encoder", "image_features"}:
            head_named.extend([(f"{name}.{n}", p) for n, p in module.named_parameters()])

    optimizer_groups += split_decay(head_named, CONFIG["lr_head"])

    optimizer = torch.optim.AdamW(optimizer_groups)
    total_steps = train_loader_len * epochs
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    return optimizer, scheduler


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device=None):
    device = DEVICE if device is None else torch.device(device)
    model.eval()
    losses = []
    preds, golds = [], []

    for batch in tqdm(loader, desc="Eval", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels,
        )
        losses.append(out["loss"].item())
        pred = out["logits"].argmax(dim=-1)
        preds.extend(pred.cpu().numpy().tolist())
        golds.extend(labels.cpu().numpy().tolist())

    metrics = {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(golds, preds),
        "f1": f1_score(golds, preds),
    }
    return metrics, np.array(golds), np.array(preds)


def train_model(model, train_loader, val_loader, epochs=3, device=None):
    device = DEVICE if device is None else torch.device(device)
    use_amp = bool(USE_AMP and device.type == "cuda")

    optimizer, scheduler = make_optimizer_and_scheduler(model, len(train_loader), epochs)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    history = []
    best_state = None
    best_f1 = -1
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        pbar = tqdm(train_loader, desc=f"Train epoch {epoch}")

        for batch in pbar:
            input_ids = batch["input_ids"].to(device, non_blocking=PIN_MEMORY)
            attention_mask = batch["attention_mask"].to(device, non_blocking=PIN_MEMORY)
            pixel_values = batch["pixel_values"].to(device, non_blocking=PIN_MEMORY)
            labels = batch["labels"].to(device, non_blocking=PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                out = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values=pixel_values,
                    labels=labels,
                )
                loss = out["loss"]

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss.item())
            pbar.set_postfix(train_loss=float(np.mean(losses[-20:])))

        val_metrics, _, _ = evaluate(model, val_loader, device=device)
        train_loss = float(np.mean(losses))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_f1": val_metrics["f1"],
        }
        history.append(row)
        print(row)

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

#### **4. Entrenamiento**


In [ ]:
print(f"Entrenando en: {DEVICE} | AMP: {USE_AMP}")


results = {}

if CONFIG["run_late_fusion"]:
    late_model = LateFusionClassifier(dropout=CONFIG["dropout"]).to(DEVICE)
    late_model, late_hist = train_model(
        late_model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=CONFIG["epochs"],
    )
    late_test_metrics, late_y_true, late_y_pred = evaluate(late_model, test_loader)
    results["LateFusion"] = {
        "history": late_hist,
        "metrics": late_test_metrics,
        "y_true": late_y_true,
        "y_pred": late_y_pred,
        "model": late_model,
    }

if CONFIG["run_mmbt"]:
    mmbt_model = MMBTLiteClassifier(
        visual_grid=CONFIG["visual_grid"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)
    mmbt_model, mmbt_hist = train_model(
        mmbt_model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=CONFIG["epochs"],
    )
    mmbt_test_metrics, mmbt_y_true, mmbt_y_pred = evaluate(mmbt_model, test_loader)
    results["MMBT-Lite"] = {
        "history": mmbt_hist,
        "metrics": mmbt_test_metrics,
        "y_true": mmbt_y_true,
        "y_pred": mmbt_y_pred,
        "model": mmbt_model,
    }

results.keys()


In [ ]:
summary_rows = []
for name, pack in results.items():
    row = {"model": name}
    row.update(pack["metrics"])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values(["f1", "acc"], ascending=False)
summary_df


In [ ]:
fig, axes = plt.subplots(1, max(1, len(results)), figsize=(6 * max(1, len(results)), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, pack) in zip(axes, results.items()):
    hist = pack["history"]
    ax.plot(hist["epoch"], hist["train_loss"], label="train_loss")
    ax.plot(hist["epoch"], hist["val_loss"], label="val_loss")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
for name, pack in results.items():
    print("=" * 80)
    print(name)
    print(pack["metrics"])
    print(classification_report(pack["y_true"], pack["y_pred"], digits=4))


In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["mismatch", "match"])
    ax.set_yticklabels(["mismatch", "match"])
    ax.set_xlabel("pred")
    ax.set_ylabel("true")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    plt.colorbar(im, ax=ax)
    plt.show()

for name, pack in results.items():
    plot_confusion(pack["y_true"], pack["y_pred"], f"Confusion matrix — {name}")


#### 5. **Visualización opcional de atención multimodal**

Esta celda intenta mostrar, de manera pedagógica, cómo el token `[CLS]` reparte atención hacia los **tokens visuales** del modelo `MMBT-Lite`.

No es una explicación causal completa del modelo, pero sí ayuda a ver que la imagen no entra como un único vector final, sino como una **secuencia de tokens visuales** que participa en el encoder.


In [ ]:
def show_mmbt_attention_example(model, ds, sample_idx=0, device=None):
    device = DEVICE if device is None else torch.device(device)
    model.eval()
    item = ds[sample_idx]
    batch = collate_fn([item])

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    pixel_values = batch["pixel_values"].to(device)

    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            output_attentions=True,
        )

    logits = out["logits"].softmax(dim=-1)[0].cpu().numpy()
    pred = logits.argmax().item()
    text_len = input_ids.shape[1]
    gh, gw = model.visual_grid
    num_visual = gh * gw

    # atención de la última capa: [B, heads, seq, seq]
    last_att = out["attentions"][-1][0]                     # [heads, seq, seq]
    cls_to_all = last_att[:, 0, :].mean(dim=0).cpu().numpy()
    cls_to_visual = cls_to_all[text_len:text_len + num_visual].reshape(gh, gw)

    img = ds.hf_split[item["hf_index"]]["image"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title("Imagen")

    axes[1].axis("off")
    axes[1].text(0, 0.7, item["caption"], wrap=True, fontsize=12)
    axes[1].set_title(f"Caption | true={item['label']} | pred={pred}")

    heat = axes[2].imshow(cls_to_visual)
    axes[2].set_title("CLS -> tokens visuales")
    axes[2].set_xticks(range(gw))
    axes[2].set_yticks(range(gh))
    plt.colorbar(heat, ax=axes[2])

    plt.tight_layout()
    plt.show()

if "MMBT-Lite" in results:
    show_mmbt_attention_example(results["MMBT-Lite"]["model"], test_ds, sample_idx=3)


#### **6. Ideas de mejora**

1. Cambiar `EfficientNet-B0` por `ConvNeXt-Tiny` o `ViT`.
2. Aumentar `visual_grid` de `(3, 3)` a `(4, 4)`.
3. Pasar de `bert-base-uncased` a `roberta-base` o `bert-large-uncased`.
4. Añadir una línea base `text-only` y otra `image-only`.
5. Reemplazar TF-IDF por **minado de negativos difíciles** con embeddings semánticos.
6. Registrar también AUC y calibración.

